# LeetCode 509 — Fibonacci Number bằng Python

**Link đề:** https://leetcode.com/problems/fibonacci-number/
**Độ khó:** Easy · **Chủ đề:** Math, Dynamic Programming, Recursion, Memoization

```python
class Solution:
    def fib(self, n: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **công thức truy hồi** (recurrence) và **điểm dừng** (base case) — nền tảng của đệ quy và quy hoạch động (DP).
2. Viết đệ quy thuần, thấy vì sao nó **bùng nổ** O(2ⁿ) bằng cách đếm số lời gọi.
3. Học **memoization** (ghi nhớ kết quả) bằng `dict` và `functools.lru_cache`.
4. Chuyển sang **DP từ dưới lên** (bottom-up) với mảng, rồi tối ưu còn **2 biến** O(1) bộ nhớ.
5. Mở rộng: **fast doubling** O(log n), công thức Binet và giới hạn số thực; so sánh 5 cách, bài tập DP cơ bản.

## 1. Đề bài

Dãy Fibonacci `F(n)`: mỗi số bằng tổng hai số đứng trước, bắt đầu từ 0 và 1:

```text
F(0) = 0, F(1) = 1
F(n) = F(n - 1) + F(n - 2),  với n > 1
```

Cho `n`, tính `F(n)`.

| n | Output | Giải thích |
|---:|---:|---|
| 2 | 1 | `F(1) + F(0) = 1 + 0` |
| 3 | 2 | `F(2) + F(1) = 1 + 1` |
| 4 | 3 | `F(3) + F(2) = 2 + 1` |

**Constraints:** `0 <= n <= 30`.

Mười số đầu: `0, 1, 1, 2, 3, 5, 8, 13, 21, 34` (ứng với `n = 0..9`).

## 2. Nền tảng

### 2.1. Tính bằng tay theo bảng

| n | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|---|
| F(n) | 0 | 1 | 1 | 2 | 3 | 5 | 8 | 13 |

Mỗi ô = tổng **hai ô liền trước**. Đây đã là ý tưởng của DP bottom-up!

### 2.2. Hàm đệ quy là gì?

Hàm **gọi lại chính nó** với input nhỏ hơn. Luôn cần:
1. **Base case**: trường hợp trả lời ngay, không gọi tiếp (ở đây `n <= 1`).
2. **Bước đệ quy**: đưa bài toán lớn về bài toán nhỏ hơn.

Ví dụ khởi động: tính giai thừa `n! = n · (n-1)!`, `0! = 1`.

In [1]:
def factorial(n):
    if n == 0:            # base case
        return 1
    return n * factorial(n - 1)

print([factorial(k) for k in range(6)])   # [1, 1, 2, 6, 24, 120]

[1, 1, 2, 6, 24, 120]


### 2.3. Gán nhiều biến cùng lúc: `a, b = b, a + b`

Python tính **toàn bộ vế phải trước** (tạo tuple `(b, a + b)` từ giá trị cũ), rồi mới gán. Nhờ vậy không cần biến tạm.

In [2]:
a, b = 0, 1
print(a, b)
a, b = b, a + b
print(a, b)     # 1 1
a, b = b, a + b
print(a, b)     # 1 2

# So sánh với cách viết sai thứ tự (gán tuần tự):
x, y = 0, 1
x = y
y = x + y       # x đã bị ghi đè -> sai
print("gán tuần tự:", x, y, "(đúng phải là 1 1)")

0 1
1 1
1 2
gán tuần tự: 1 2 (đúng phải là 1 1)


## 3. Cách 1 — Đệ quy thuần (dịch thẳng công thức)

In [3]:
def fib_recursive(n):
    if n <= 1:
        return n
    return fib_recursive(n - 1) + fib_recursive(n - 2)

print([fib_recursive(k) for k in range(10)])

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


### Nút thắt: tính đi tính lại cùng một giá trị

Cây lời gọi của `F(5)`:

```text
                    F(5)
              /             \
          F(4)               F(3)
        /      \            /    \
     F(3)      F(2)      F(2)    F(1)
    /   \      /  \      /  \
  F(2) F(1) F(1) F(0) F(1) F(0)
  /  \
F(1) F(0)
```
`F(3)` bị tính **2 lần**, `F(2)` **3 lần**... Số lời gọi tăng theo cấp số nhân (~1.618ⁿ). Ta đếm thật:

In [4]:
from collections import Counter

def count_calls(n):
    calls = Counter()
    def f(k):
        calls[k] += 1
        if k <= 1:
            return k
        return f(k - 1) + f(k - 2)
    f(n)
    return calls

c5 = count_calls(5)
print("F(5): số lần gọi mỗi k:", dict(sorted(c5.items(), reverse=True)), "| tổng:", sum(c5.values()))
for n in [10, 20, 25, 30]:
    print(f"n={n}: tổng số lời gọi = {sum(count_calls(n).values()):,}")

F(5): số lần gọi mỗi k: {5: 1, 4: 1, 3: 2, 2: 3, 1: 5, 0: 3} | tổng: 15
n=10: tổng số lời gọi = 177
n=20: tổng số lời gọi = 21,891
n=25: tổng số lời gọi = 242,785


n=30: tổng số lời gọi = 2,692,537


Với `n = 30` cần **hơn 2,6 triệu** lời gọi chỉ để ra một số. Vẫn chạy được trong giới hạn của đề (`n ≤ 30`) nhưng rất lãng phí, và với `n = 50` thì gần như không chạy nổi.

## 4. Insight — mỗi `F(k)` chỉ cần tính **một lần**

Chỉ có `n + 1` giá trị khác nhau `F(0)..F(n)`. Nếu **ghi nhớ** kết quả mỗi lần tính xong, lần sau chỉ việc tra lại → tổng công việc `O(n)`.

Hai cách hiện thực:
- **Top-down + memo**: giữ đệ quy, thêm "cuốn sổ" `memo`.
- **Bottom-up**: tính từ nhỏ đến lớn `F(0), F(1), F(2), ...` — không cần đệ quy.

## 5. Cách 2 — Memoization (top-down)

### 5.1. Tự viết memo bằng `dict`

In [5]:
def fib_memo(n, memo=None):
    if memo is None:
        memo = {}
    if n <= 1:
        return n
    if n in memo:                     # đã tính rồi -> tra sổ
        return memo[n]
    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]

print([fib_memo(k) for k in range(10)], fib_memo(30), fib_memo(90))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34] 832040 2880067194370816120


Vì sao `memo=None` rồi mới tạo `{}` bên trong? Nếu viết `memo={}` làm giá trị mặc định, **cùng một dict** được dùng lại giữa các lần gọi (default argument chỉ tạo một lần) — ở bài này vô hại, nhưng là cái bẫy kinh điển khi giá trị mặc định là list/dict.

### 5.2. `functools.lru_cache` — memo có sẵn

Decorator `@lru_cache(maxsize=None)` tự động ghi nhớ kết quả theo tham số.

In [6]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib_cache(n):
    if n <= 1:
        return n
    return fib_cache(n - 1) + fib_cache(n - 2)

print(fib_cache(30), fib_cache(100))
print(fib_cache.cache_info())

832040 354224848179261915075
CacheInfo(hits=99, misses=101, maxsize=None, currsize=101)


### 5.3. Giới hạn của top-down: độ sâu đệ quy

Mỗi lời gọi đệ quy chiếm một "khung" trên **call stack**. Python giới hạn độ sâu mặc định khoảng 1000 (`sys.getrecursionlimit()`). Memo top-down với `n = 5000` (gọi lần đầu) sẽ đi sâu ~5000 tầng → `RecursionError`. Bottom-up **không** gặp vấn đề này — thêm một lý do để ưu tiên nó.

In [7]:
import sys
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
try:
    fib_memo(5000)
except RecursionError:
    print("fib_memo(5000) -> RecursionError (đệ quy quá sâu)")
a, b = 0, 1
for _ in range(5000):          # bottom-up (sẽ học kỹ ở mục 6–7)
    a, b = b, a + b
print("bottom-up F(5000) có", len(str(a)), "chữ số — không lỗi")

giới hạn đệ quy mặc định: 1000
fib_memo(5000) -> RecursionError (đệ quy quá sâu)
bottom-up F(5000) có 1045 chữ số — không lỗi


## 6. Cách 3 — DP bottom-up với mảng

1. `dp = [0] * (n + 1)`; `dp[1] = 1` (nếu `n ≥ 1`).
2. `for i in range(2, n + 1): dp[i] = dp[i-1] + dp[i-2]`.
3. Trả `dp[n]`.

In [8]:
def fib_table(n):
    if n <= 1:
        return n
    dp = [0] * (n + 1)
    dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]

print([fib_table(k) for k in range(10)])

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


## 7. Cách 4 — Lời giải chính: chỉ giữ 2 biến (O(1) bộ nhớ)

Nhận xét: `dp[i]` chỉ cần `dp[i-1]` và `dp[i-2]` → không cần cả mảng. Giữ:
- `a` = `F(i)`, `b` = `F(i+1)`.

Mỗi bước: `a, b = b, a + b` (tiến `i` thêm 1). Lặp `n` lần từ `(F(0), F(1)) = (0, 1)` thì `a = F(n)`.

In [9]:
def fib_two_vars(n):
    a, b = 0, 1           # a = F(0), b = F(1)
    for _ in range(n):
        a, b = b, a + b   # (F(i), F(i+1)) -> (F(i+1), F(i+2))
    return a

print([fib_two_vars(k) for k in range(10)])

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


Biến `_` là quy ước cho "biến vòng lặp không dùng tới".

## 8. TRACE từng bước

`n = 6`:

| vòng | a trước | b trước | a sau (= b cũ) | b sau (= a + b cũ) | ý nghĩa |
|---:|---:|---:|---:|---:|---|
| 1 | 0 | 1 | 1 | 1 | (F1, F2) |
| 2 | 1 | 1 | 1 | 2 | (F2, F3) |
| 3 | 1 | 2 | 2 | 3 | (F3, F4) |
| 4 | 2 | 3 | 3 | 5 | (F4, F5) |
| 5 | 3 | 5 | 5 | 8 | (F5, F6) |
| 6 | 5 | 8 | 8 | 13 | (F6, F7) |

Kết thúc: `a = 8 = F(6)`.

In [10]:
def trace_fib(n):
    a, b = 0, 1
    print(f"khởi tạo: a=F(0)={a}, b=F(1)={b}")
    for i in range(1, n + 1):
        a_old, b_old = a, b
        a, b = b, a + b
        print(f"vòng {i}: ({a_old}, {b_old}) -> ({a}, {b})   tức (F({i}), F({i + 1}))")
    print(f"=> F({n}) = {a}")
    return a

trace_fib(6)
print()
trace_fib(0)

khởi tạo: a=F(0)=0, b=F(1)=1
vòng 1: (0, 1) -> (1, 1)   tức (F(1), F(2))
vòng 2: (1, 1) -> (1, 2)   tức (F(2), F(3))
vòng 3: (1, 2) -> (2, 3)   tức (F(3), F(4))
vòng 4: (2, 3) -> (3, 5)   tức (F(4), F(5))
vòng 5: (3, 5) -> (5, 8)   tức (F(5), F(6))
vòng 6: (5, 8) -> (8, 13)   tức (F(6), F(7))
=> F(6) = 8

khởi tạo: a=F(0)=0, b=F(1)=1
=> F(0) = 0


0

Với `n = 0` vòng lặp không chạy, trả `a = 0` — đúng mà không cần `if` riêng.

Trace phiên bản **memo** để thấy mỗi `F(k)` được tính đúng một lần:

In [11]:
def trace_memo(n):
    memo = {}
    def f(k, depth):
        pad = "  " * depth
        if k <= 1:
            print(f"{pad}F({k}) = {k} (base case)")
            return k
        if k in memo:
            print(f"{pad}F({k}) = {memo[k]} (tra memo)")
            return memo[k]
        print(f"{pad}F({k}) -> tính F({k-1}) + F({k-2})")
        memo[k] = f(k - 1, depth + 1) + f(k - 2, depth + 1)
        print(f"{pad}F({k}) = {memo[k]} (lưu memo)")
        return memo[k]
    return f(n, 0)

trace_memo(5)

F(5) -> tính F(4) + F(3)
  F(4) -> tính F(3) + F(2)
    F(3) -> tính F(2) + F(1)
      F(2) -> tính F(1) + F(0)
        F(1) = 1 (base case)
        F(0) = 0 (base case)
      F(2) = 1 (lưu memo)
      F(1) = 1 (base case)
    F(3) = 2 (lưu memo)
    F(2) = 1 (tra memo)
  F(4) = 3 (lưu memo)
  F(3) = 2 (tra memo)
F(5) = 5 (lưu memo)


5

## 9. Cách 5 (nâng cao) — Fast doubling O(log n)

Hai đẳng thức (chứng minh được bằng ma trận `[[1,1],[1,0]]^n`):

```text
F(2k)   = F(k) · (2·F(k+1) − F(k))
F(2k+1) = F(k)² + F(k+1)²
```
Từ `(F(k), F(k+1))` suy ra `(F(2k), F(2k+1))` → mỗi bước **gấp đôi** chỉ số, nên chỉ cần `O(log n)` bước. Rất hữu ích khi `n` cực lớn (ví dụ `10^18` kèm modulo).

In [12]:
def fib_fast_doubling(n):
    def pair(k):                       # trả (F(k), F(k+1))
        if k == 0:
            return (0, 1)
        a, b = pair(k // 2)            # a = F(m), b = F(m+1), m = k // 2
        c = a * (2 * b - a)            # F(2m)
        d = a * a + b * b              # F(2m+1)
        return (d, c + d) if k % 2 else (c, d)
    return pair(n)[0]

print([fib_fast_doubling(k) for k in range(10)])
print(fib_fast_doubling(300))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
222232244629420445529739893461909967206666939096499764990979600


### Bonus — Công thức Binet (và vì sao không nên dùng số thực)

`F(n) = (φⁿ − ψⁿ) / √5` với `φ = (1+√5)/2`, `ψ = (1−√5)/2`. Đẹp về toán, nhưng float có sai số: với `n` lớn (khoảng `n > 70`) kết quả làm tròn bị sai.

In [13]:
import math
def fib_binet(n):
    phi = (1 + math.sqrt(5)) / 2
    psi = (1 - math.sqrt(5)) / 2
    return round((phi**n - psi**n) / math.sqrt(5))

print("n<=30 khớp:", all(fib_binet(k) == fib_two_vars(k) for k in range(31)))
first_bad = next(k for k in range(200) if fib_binet(k) != fib_two_vars(k))
print("n đầu tiên Binet (float) sai:", first_bad)

n<=30 khớp: True
n đầu tiên Binet (float) sai: 71


## 10. Code nộp LeetCode

### Bản 2 biến (khuyên dùng)

In [14]:
class Solution:
    def fib(self, n: int) -> int:
        a, b = 0, 1
        for _ in range(n):
            a, b = b, a + b
        return a

s = Solution()
print(s.fib(2), s.fib(3), s.fib(4))   # 1 2 3

1 2 3


### Bản memo (top-down)

In [15]:
class SolutionMemo:
    def fib(self, n: int) -> int:
        memo = {0: 0, 1: 1}
        def f(k):
            if k not in memo:
                memo[k] = f(k - 1) + f(k - 2)
            return memo[k]
        return f(n)

print(SolutionMemo().fib(30))

832040


(Khi nộp, đổi tên lớp thành `Solution`.)

## 11. Vì sao đúng?

**Cách 2 biến — invariant:** trước vòng thứ `i` (đếm từ 0), `a = F(i)` và `b = F(i+1)`.
- Ban đầu `i = 0`: `a = 0 = F(0)`, `b = 1 = F(1)` ✔.
- Giữ nguyên qua một vòng: `a_mới = b = F(i+1)`, `b_mới = a + b = F(i) + F(i+1) = F(i+2)` ✔ (theo công thức truy hồi).
- Sau `n` vòng: `a = F(n)`.

**Memo:** mọi giá trị trong `memo` đều được tính bằng công thức đúng; tra lại chỉ trả về đúng giá trị đã tính → kết quả giống đệ quy thuần, chỉ khác số lần tính.

## 12. Độ phức tạp

| Cách | Time | Space | Ghi chú |
|---|---:|---:|---|
| Đệ quy thuần | `O(φⁿ)` ≈ `O(1.618ⁿ)` | `O(n)` stack | 2,6 triệu lời gọi với n = 30 |
| Memo (dict / lru_cache) | `O(n)` | `O(n)` memo + stack | top-down |
| DP mảng | `O(n)` | `O(n)` | bottom-up |
| 2 biến | `O(n)` | `O(1)` | ⭐ chuẩn phỏng vấn |
| Fast doubling | `O(log n)` phép nhân | `O(log n)` stack | dùng khi n rất lớn |
| Binet (float) | `O(1)` | `O(1)` | sai khi n lớn |

(Phân tích trên coi phép cộng số nguyên là O(1); thật ra `F(n)` có ~`0.7n` bit nên với `n` rất lớn phép cộng tốn thêm.)

In [16]:
import time
for f in [fib_recursive, fib_memo, fib_table, fib_two_vars, fib_fast_doubling]:
    t0 = time.perf_counter(); v = f(27); t1 = time.perf_counter()
    print(f"{f.__name__:18s}: F(27) = {v}  ({(t1 - t0) * 1000:.3f} ms)")

fib_recursive     : F(27) = 196418  (23.704 ms)
fib_memo          : F(27) = 196418  (0.009 ms)
fib_table         : F(27) = 196418  (0.005 ms)
fib_two_vars      : F(27) = 196418  (0.002 ms)
fib_fast_doubling : F(27) = 196418  (0.004 ms)


## 13. Bộ test

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

s = Solution()
check("đề 1: n=2", s.fib(2), 1)
check("đề 2: n=3", s.fib(3), 2)
check("đề 3: n=4", s.fib(4), 3)
check("n=0", s.fib(0), 0)
check("n=1", s.fib(1), 1)
check("n=10", s.fib(10), 55)
check("n=30 (cực đại)", s.fib(30), 832040)

PASS đề 1: n=2: 1
PASS đề 2: n=3: 2
PASS đề 3: n=4: 3
PASS n=0: 0
PASS n=1: 1
PASS n=10: 55
PASS n=30 (cực đại): 832040


In [18]:
impls = [fib_recursive, fib_memo, fib_cache, fib_table, fib_two_vars, fib_fast_doubling,
         Solution().fib, SolutionMemo().fib]
for n in range(31):
    values = {f(n) for f in impls}
    assert len(values) == 1, (n, values)
# tính chất: F(n+1)·F(n-1) − F(n)² = (−1)^n  (đẳng thức Cassini)
for n in range(1, 200):
    assert fib_two_vars(n + 1) * fib_two_vars(n - 1) - fib_two_vars(n) ** 2 == (-1) ** n
print("n = 0..30: 8 cài đặt khớp nhau; đẳng thức Cassini đúng với n < 200 — PASS")

n = 0..30: 8 cài đặt khớp nhau; đẳng thức Cassini đúng với n < 200 — PASS


## 14. Lỗi người mới hay gặp

### Lỗi 1: thiếu base case `n == 0`
```python
def f(n):
    if n == 1: return 1
    return f(n-1) + f(n-2)     # f(0) -> f(-1) -> ... không bao giờ dừng
```
→ `RecursionError`.

### Lỗi 2: gán tuần tự thay vì gán đồng thời
```python
a = b
b = a + b                      # a đã là b mới -> b = 2b
```

### Lỗi 3: lặp sai số lần (off-by-one)
`for _ in range(n - 1)` → trả `F(n-1)`.

### Lỗi 4: đệ quy thuần với n lớn
Đúng nhưng chậm theo cấp số nhân; với `n = 40` đã mất nhiều giây.

In [19]:
def buggy_base(n):
    if n == 1:
        return 1
    return buggy_base(n - 1) + buggy_base(n - 2)

try:
    buggy_base(3)
except RecursionError as e:
    print("thiếu base case -> RecursionError:", str(e)[:45])

def buggy_sequential(n):
    a, b = 0, 1
    for _ in range(n):
        a = b
        b = a + b
    return a

def buggy_off_by_one(n):
    a, b = 0, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return a

print("gán tuần tự :", [buggy_sequential(k) for k in range(8)])
print("lệch 1      :", [buggy_off_by_one(k) for k in range(8)])
print("đúng        :", [fib_two_vars(k) for k in range(8)])

thiếu base case -> RecursionError: maximum recursion depth exceeded
gán tuần tự : [0, 1, 2, 4, 8, 16, 32, 64]
lệch 1      : [0, 0, 1, 1, 2, 3, 5, 8]
đúng        : [0, 1, 1, 2, 3, 5, 8, 13]


## 15. Mẹo nhận dạng pattern

- Đề cho **công thức truy hồi** `f(n)` theo `f(n-1), f(n-2), ...` → DP.
- Đệ quy thuần có **bài toán con lặp lại** (overlapping subproblems) → thêm **memo** hoặc chuyển **bottom-up**.
- `f(n)` chỉ phụ thuộc **k giá trị trước** → giữ **k biến** (rolling variables), bộ nhớ O(1).
- Các bài "anh em" của Fibonacci: Climbing Stairs (70), Tribonacci (1137), Min Cost Climbing Stairs (746), House Robber (198).
- `n` cực lớn + modulo → ma trận / fast doubling O(log n).

## 16. Biến thể

### 16.1. Climbing Stairs (LeetCode 70)
Mỗi lần leo 1 hoặc 2 bậc, hỏi có bao nhiêu cách lên `n` bậc → `ways(n) = ways(n-1) + ways(n-2)`, `ways(1)=1, ways(2)=2` → chính là `F(n+1)`.

In [20]:
def climb_stairs(n):
    a, b = 1, 1          # ways(0)=1 (đứng yên), ways(1)=1
    for _ in range(n):
        a, b = b, a + b
    return a

print([climb_stairs(k) for k in range(1, 8)])
print(all(climb_stairs(k) == fib_two_vars(k + 1) for k in range(30)))

[1, 2, 3, 5, 8, 13, 21]
True


### 16.2. Tribonacci (LeetCode 1137)
`T(0)=0, T(1)=1, T(2)=1, T(n)=T(n-1)+T(n-2)+T(n-3)` → giữ 3 biến.

In [21]:
def tribonacci(n):
    a, b, c = 0, 1, 1
    for _ in range(n):
        a, b, c = b, c, a + b + c
    return a

print([tribonacci(k) for k in range(10)])   # [0, 1, 1, 2, 4, 7, 13, 24, 44, 81]

[0, 1, 1, 2, 4, 7, 13, 24, 44, 81]


## 17. Bài tập (đáp án ở cuối)

### Bài 1 — Đoán output
Sau đoạn sau, `a, b` bằng bao nhiêu?
```python
a, b = 0, 1
for _ in range(5):
    a, b = b, a + b
```

### Bài 2 — Điền TODO (2 biến)

In [22]:
def exercise_fib(n):
    # TODO 1: khởi tạo a = F(0), b = F(1)
    # TODO 2: lặp n lần, gán đồng thời
    # TODO 3: trả a
    pass

# assert [exercise_fib(k) for k in range(10)] == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

### Bài 3 — Danh sách n số Fibonacci đầu tiên

In [23]:
def fib_list(n):
    # TODO: trả list [F(0), ..., F(n-1)]
    pass

# assert fib_list(0) == []
# assert fib_list(7) == [0, 1, 1, 2, 3, 5, 8]

### Bài 4 — Fibonacci modulo
Tính `F(n) % m` cho `n` tới `10^6` mà các số trung gian luôn nhỏ.

In [24]:
def fib_mod(n, m):
    # TODO
    pass

# assert fib_mod(10, 7) == 55 % 7
# assert fib_mod(10**6, 10**9 + 7) == 918091266

### Bài 5 — Min Cost Climbing Stairs (LeetCode 746)
`cost[i]` là chi phí bước lên bậc `i`; xuất phát ở bậc 0 hoặc 1, mỗi lần leo 1 hoặc 2 bậc, đích là **vượt qua** bậc cuối. Tìm chi phí nhỏ nhất.

In [25]:
def min_cost_climbing(cost):
    # TODO
    pass

# assert min_cost_climbing([10, 15, 20]) == 15
# assert min_cost_climbing([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]) == 6

### Bài 6 — Mini challenge: số Fibonacci đầu tiên có ≥ d chữ số
Trả về chỉ số `n` nhỏ nhất sao cho `F(n)` có ít nhất `d` chữ số (Project Euler 25).

In [26]:
def first_fib_with_digits(d):
    # TODO
    pass

# assert first_fib_with_digits(3) == 12      # F(12) = 144
# assert first_fib_with_digits(1000) == 4782

## 18. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Mỗi vòng `(a, b)` tiến 1 bước | sau `k` vòng `a = F(k)` | kiểm bằng `trace_fib(5)` |
| 2 | Xem mục 7 | `a, b = 0, 1` | `for _ in range(n): a, b = b, a + b` |
| 3 | Như bài 2 nhưng ghi lại `a` | `out.append(a)` **trước** khi cập nhật | lặp `n` lần |
| 4 | Lấy modulo sau mỗi phép cộng | `(a + b) % m` | `a, b = b, (a + b) % m` |
| 5 | `dp[i]` = chi phí nhỏ nhất để **đứng** ở bậc `i` | `dp[i] = min(dp[i-1] + cost[i-1], dp[i-2] + cost[i-2])` | chỉ cần 2 biến |
| 6 | Duyệt tăng dần | `len(str(a)) >= d` | giữ biến đếm chỉ số |

## 19. Đáp án tham khảo

**Bài 1:** sau 5 vòng `a = F(5) = 5`, `b = F(6) = 8`.

In [27]:
a, b = 0, 1
for _ in range(5):
    a, b = b, a + b
print(a, b)

5 8


In [28]:
def answer_exercise_fib(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

def answer_fib_list(n):
    out = []
    a, b = 0, 1
    for _ in range(n):
        out.append(a)
        a, b = b, a + b
    return out

def answer_fib_mod(n, m):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, (a + b) % m
    return a % m

def answer_min_cost_climbing(cost):
    prev2, prev1 = 0, 0          # chi phí để đứng ở bậc i-2, i-1 (bắt đầu: bậc 0 và 1 miễn phí)
    for i in range(2, len(cost) + 1):
        prev2, prev1 = prev1, min(prev1 + cost[i - 1], prev2 + cost[i - 2])
    return prev1

def answer_first_fib_with_digits(d):
    a, b, n = 0, 1, 0
    while len(str(a)) < d:
        a, b = b, a + b
        n += 1
    return n

assert [answer_exercise_fib(k) for k in range(10)] == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
assert answer_fib_list(0) == [] and answer_fib_list(7) == [0, 1, 1, 2, 3, 5, 8]
assert answer_fib_mod(10, 7) == 55 % 7
assert answer_fib_mod(10**6, 10**9 + 7) == fib_fast_doubling(10**6) % (10**9 + 7) == 918091266
assert answer_min_cost_climbing([10, 15, 20]) == 15
assert answer_min_cost_climbing([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]) == 6
assert answer_first_fib_with_digits(3) == 12
assert answer_first_fib_with_digits(1000) == 4782
print("Đáp án bài 2–6: PASS toàn bộ test!")

Đáp án bài 2–6: PASS toàn bộ test!


## 20. Template nhớ nhanh + checklist submit

```python
# DP 2 biến cho truy hồi bậc 2
a, b = F0, F1
for _ in range(n):
    a, b = b, a + b      # gán ĐỒNG THỜI
return a
```

Lộ trình tư duy DP: **Đệ quy (đúng nhưng chậm) → Memo (nhớ kết quả) → Bottom-up (mảng) → Rolling variables (O(1)).**

### Checklist
- [ ] Base case `F(0) = 0`, `F(1) = 1` đúng; `n = 0` trả `0`.
- [ ] Gán đồng thời `a, b = b, a + b`.
- [ ] Lặp đúng `n` lần (kiểm tra với `n = 0, 1, 2`).
- [ ] Không dùng đệ quy thuần khi `n` lớn; không dùng Binet (float) khi `n` lớn.
- [ ] Time `O(n)`, Space `O(1)`.